In [22]:
import pandas as pd
import json
from pathlib import Path
from pprint import pprint
import matplotlib.pyplot as plt
import seaborn as sns
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import os

In [23]:
#Load Dyanmic Paths
BASE_DIR = Path.cwd()
EMBED_DIR = BASE_DIR / "Embeddings"
IR_DIR = BASE_DIR / "IR2025"

DOCS_CSV = IR_DIR / "documents.csv"
QUERIES_CSV = IR_DIR / "queries.csv"
DOC_EMBED_FILE = EMBED_DIR / "ir2025_embeddings.npy"
QUERY_EMBED_FILE = EMBED_DIR / "query_embeddings.npy"
DOC_INDEXED_CSV = IR_DIR / "documents_with_index.csv"
QUERY_INDEXED_CSV = IR_DIR / "queries_with_index.csv"

EMBED_DIR.mkdir(exist_ok=True)
IR_DIR.mkdir(exist_ok=True)

In [24]:
#Preprocess function
def preprocess(s):
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = re.sub(r"\s+", " ", s)                      # collapse spaces/newlines
    s = re.sub(r"http\S+|www\.\S+", "<URL>", s)     # replace URLs
    s = re.sub(r"\S+@\S+", "<EMAIL>", s)            # replace emails
    s = re.sub(r"<[^>]+>", " ", s)                  # remove HTML tags
    s = ''.join(ch for ch in s if ord(ch) >= 32)    # remove control chars
    s = re.sub(r"['\"]", "", s)                     # remove quotes
    return s.strip()


In [25]:
#Load docs/queries and apply preprocess both
df_docs = pd.read_csv(DOCS_CSV)
df_queries = pd.read_csv(QUERIES_CSV)

df_docs = df_docs.dropna(subset=["Text"])
df_docs["Text"] = df_docs["Text"].astype(str).map(preprocess)

df_queries = df_queries.dropna(subset=["Text"])
df_queries["Text"] = df_queries["Text"].astype(str).map(preprocess)

In [26]:
#Select model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [27]:
#Class for embedding generation and indexing
class EmbeddingIndexer:
    def __init__(self, model, text_column="Text",
                 embedding_file="Embeddings/ir2025_embeddings.npy",
                 indexed_csv="IR2025/documents_with_index.csv",
                 batch_size=64,
                 use_cosine=True):
        self.model = model
        self.text_column = text_column
        self.embedding_file = embedding_file
        self.indexed_csv = indexed_csv
        self.batch_size = batch_size
        self.use_cosine = use_cosine
        self.index = None 

    #Method for embedding generation
    def create_embeddings(self, texts, show_progress=True):
        if isinstance(texts, pd.Series):
            texts = texts.astype(str).tolist()
        elif isinstance(texts, str):
            texts = [texts]
        elif not isinstance(texts, list):
            raise TypeError("Input must be a list, string, or pandas Series.")

        print(f"Encoding {len(texts)} text(s) using {self.model.__class__.__name__}...")
        embeddings = self.model.encode(
            texts,
            batch_size=self.batch_size,
            show_progress_bar=show_progress,
            convert_to_numpy=True
        )

        print(f"Embeddings created successfully! Shape: {embeddings.shape}")
        return embeddings

    #Method for indexing embeddings
    def index_embeddings(self, df):
        print(f"Generating embeddings for DataFrame with {len(df)} rows...")
        embeddings = self.create_embeddings(df[self.text_column], show_progress=True)

        np.save(self.embedding_file, embeddings)
        df["embedding_index"] = np.arange(len(df))
        df.to_csv(self.indexed_csv, index=False)

        print(f"Saved embeddings → {self.embedding_file}")
        print(f"Saved indexed CSV → {self.indexed_csv}")
        return df, embeddings

    #Method for building FAISS index
    def build_faiss_index(self, embeddings):
        embeddings = embeddings.astype("float32")

        if self.use_cosine:
            faiss.normalize_L2(embeddings)
            index = faiss.IndexFlatIP(embeddings.shape[1])
        else:
            index = faiss.IndexFlatL2(embeddings.shape[1])

        index.add(embeddings)
        print(f"FAISS index built — {index.ntotal} vectors ({'cosine' if self.use_cosine else 'L2'})")
        return index

    #Load or create embeddings, then always index them.
    def build_index(self, df):
        if os.path.exists(self.embedding_file):
            print(f"Loading existing embeddings...")
            embeddings = np.load(self.embedding_file)
        else:
            df, embeddings = self.index_embeddings(df)

        df["embedding_index"] = np.arange(len(embeddings))
        self.index = self.build_faiss_index(embeddings)
        return df, embeddings



In [28]:
#Initialize Faiss index
indexer = EmbeddingIndexer(model)

#Build index 
df, embeddings = indexer.build_index(df_docs)


Loading existing embeddings...
FAISS index built — 18316 vectors (cosine)


In [29]:
#Create query embeddings
query_embeddings = indexer.create_embeddings(df_queries["Text"], show_progress=False)
faiss.normalize_L2(query_embeddings)


Encoding 10 text(s) using SentenceTransformer...
Embeddings created successfully! Shape: (10, 384)
